In [55]:
import requests
import os
from urllib.parse import urlparse
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
import json
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
from sklearn.metrics import mean_squared_error

In [3]:
def download_file(url, save_path=None):
    try:
        response = requests.get(url, stream=True, timeout=15)
        response.raise_for_status()

        if save_path is None:
            filename = os.path.basename(urlparse(url).path) or "downloaded_file"
            save_path = os.path.join(os.getcwd(), filename)

        total_size = int(response.headers.get('content-length', 0))
        block_size = 1024

        downloaded_size = 0
        with open(save_path, 'wb') as file:
            for data in response.iter_content(block_size):
                file.write(data)
                downloaded_size += len(data)
                if total_size:
                    percent = (downloaded_size / total_size) * 100
                    print(f"\rDownloading: {percent:.2f}%", end="")

        print(f"\nDownload complete: {save_path}")

    except requests.exceptions.MissingSchema:
        print("Invalid URL format.")
    except requests.exceptions.ConnectionError:
        print("Connection error. Check your internet connection.")
    except requests.exceptions.Timeout:
        print("Request timed out.")
    except requests.exceptions.HTTPError as e:
        print(f"HTTP error: {e}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")


In [4]:
BASE_URL = 'https://raw.githubusercontent.com/jcpeterson/choices13k/refs/heads/main'

csv_url = f'{BASE_URL}/c13k_selections.csv'
json_url = f'{BASE_URL}/c13k_problems.json'

if not os.path.exists('c13k_selections.csv'):
    download_file(csv_url)
if not os.path.exists('c13k_problems.json'):
    download_file(json_url)

In [5]:
def validate_json_file(path: str) -> bool:
    try:
        with open(path, "r", encoding="utf-8") as f:
            json.load(f)
        print("JSON is valid")
        return True
    except json.JSONDecodeError as e:
        print("Invalid JSON:", e)
        return False

validate_json_file("c13k_problems.json")


JSON is valid


True

In [6]:
df = pd.read_csv('c13k_selections.csv')
df.head()
df.isnull().sum()

Problem      0
Feedback     0
n            0
Block        0
Ha           0
pHa          0
La           0
Hb           0
pHb          0
Lb           0
LotShapeB    0
LotNumB      0
Amb          0
Corr         0
bRate        0
bRate_std    0
dtype: int64

In [7]:
len(df)

14568

In [8]:
problems = pd.read_json("c13k_problems.json", orient='index')
problems

,B,A
0,"[[0.9500000000000001, 21.0], [0.05, 23.0]]","[[0.9500000000000001, 26.0], [0.05, -1.0]]"
1,"[[0.75, -5.0], [0.25, 8.0]]","[[0.6000000000000001, 14.0], [0.4, -18.0]]"
2,"[[1.0, 1.0]]","[[0.5, 2.0], [0.5, 0.0]]"
3,"[[0.75, -31.0], [0.125, 86.5], [0.125, 87.5]]","[[0.05, 37.0], [0.9500000000000001, 8.0]]"
4,"[[0.25, -36.0], [0.375, 41.0], [0.1875, 43.0],...","[[1.0, 26.0], [0.0, 26.0]]"
...,...,...
14563,"[[0.199999999999999, 0.0], [0.8, 42.0]]","[[1.0, 30.0], [0.0, 30.0]]"
14564,"[[0.199999999999999, 7.0], [0.8, 18.0]]","[[0.5, 70.0], [0.5, -42.0]]"
14565,"[[0.6000000000000001, -34.0], [0.0125, 28.5], ...","[[0.4, 8.0], [0.6000000000000001, -17.0]]"
14566,"[[0.5, -12.0], [0.5, 45.0]]","[[0.5, 89.0], [0.5, -49.0]]"


In [9]:
selections = pd.read_csv('c13k_selections.csv')

with pd.option_context('display.max_columns', None):
    display(selections)

,Problem,Feedback,n,Block,Ha,pHa,La,Hb,pHb,Lb,LotShapeB,LotNumB,Amb,Corr,bRate,bRate_std
0,1,True,15,2,26,0.95,-1,23,0.05,21,0,1,False,0,0.626667,0.384460
1,2,True,15,4,14,0.60,-18,8,0.25,-5,0,1,True,-1,0.493333,0.413118
2,3,True,17,4,2,0.50,0,1,1.00,1,0,1,False,0,0.611765,0.432843
3,4,True,18,3,37,0.05,8,87,0.25,-31,1,2,False,0,0.222222,0.387383
4,5,False,15,1,26,1.00,26,45,0.75,-36,2,5,False,0,0.586667,0.450185
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14563,13002,True,15,3,30,1.00,30,42,0.80,0,0,1,True,0,0.367619,0.302731
14564,13003,True,15,5,70,0.50,-42,18,0.80,7,0,1,False,0,0.760000,0.364104
14565,13004,True,15,5,8,0.40,-17,31,0.40,-34,1,6,False,0,0.666667,0.367747
14566,13005,True,15,2,89,0.50,-49,45,0.50,-12,0,1,False,0,0.386667,0.381476


In [10]:
selections.columns = [col.lower() for col in selections.columns]

In [11]:
df_full = selections.join(problems, how='left')

In [12]:
with pd.option_context('display.max_columns', None):
    display(df_full)

,problem,feedback,n,block,ha,pha,la,hb,phb,lb,lotshapeb,lotnumb,amb,corr,brate,brate_std,B,A
0,1,True,15,2,26,0.95,-1,23,0.05,21,0,1,False,0,0.626667,0.384460,"[[0.9500000000000001, 21.0], [0.05, 23.0]]","[[0.9500000000000001, 26.0], [0.05, -1.0]]"
1,2,True,15,4,14,0.60,-18,8,0.25,-5,0,1,True,-1,0.493333,0.413118,"[[0.75, -5.0], [0.25, 8.0]]","[[0.6000000000000001, 14.0], [0.4, -18.0]]"
2,3,True,17,4,2,0.50,0,1,1.00,1,0,1,False,0,0.611765,0.432843,"[[1.0, 1.0]]","[[0.5, 2.0], [0.5, 0.0]]"
3,4,True,18,3,37,0.05,8,87,0.25,-31,1,2,False,0,0.222222,0.387383,"[[0.75, -31.0], [0.125, 86.5], [0.125, 87.5]]","[[0.05, 37.0], [0.9500000000000001, 8.0]]"
4,5,False,15,1,26,1.00,26,45,0.75,-36,2,5,False,0,0.586667,0.450185,"[[0.25, -36.0], [0.375, 41.0], [0.1875, 43.0],...","[[1.0, 26.0], [0.0, 26.0]]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14563,13002,True,15,3,30,1.00,30,42,0.80,0,0,1,True,0,0.367619,0.302731,"[[0.199999999999999, 0.0], [0.8, 42.0]]","[[1.0, 30.0], [0.0, 30.0]]"
14564,13003,True,15,5,70,0.50,-42,18,0.80,7,0,1,False,0,0.760000,0.364104,"[[0.199999999999999, 7.0], [0.8, 18.0]]","[[0.5, 70.0], [0.5, -42.0]]"
14565,13004,True,15,5,8,0.40,-17,31,0.40,-34,1,6,False,0,0.666667,0.367747,"[[0.6000000000000001, -34.0], [0.0125, 28.5], ...","[[0.4, 8.0], [0.6000000000000001, -17.0]]"
14566,13005,True,15,2,89,0.50,-49,45,0.50,-12,0,1,False,0,0.386667,0.381476,"[[0.5, -12.0], [0.5, 45.0]]","[[0.5, 89.0], [0.5, -49.0]]"


# Investigate duplicate values

In [13]:
counts = df_full.groupby('problem').size().reset_index(name='count').sort_values(by="count", ascending=False)

In [14]:
idx = counts.loc[counts["count"] > 1, "problem"].tolist()


In [15]:
len(idx)
idx


[2675,
 4755,
 6222,
 1503,
 5618,
 1505,
 2679,
 2680,
 1508,
 2681,
 2682,
 1511,
 7,
 8,
 6260,
 4759,
 4760,
 1515,
 4761,
 4762,
 1518,
 1519,
 5133,
 5134,
 1522,
 3565,
 3566,
 3567,
 1526,
 5135,
 3569,
 1529,
 6304,
 1531,
 2696,
 4260,
 3572,
 1535,
 5926,
 4262,
 5927,
 1539,
 5623,
 5140,
 4266,
 10,
 47,
 1543,
 49,
 1544,
 1545,
 5409,
 4772,
 4269,
 2708,
 2709,
 57,
 58,
 3582,
 3583,
 61,
 4270,
 5142,
 4774,
 65,
 5624,
 67,
 5794,
 69,
 1558,
 5412,
 4778,
 4779,
 5146,
 6225,
 76,
 1564,
 1565,
 79,
 3594,
 5414,
 4783,
 4784,
 3598,
 5627,
 4786,
 1573,
 2729,
 3601,
 4787,
 4788,
 92,
 1578,
 94,
 5416,
 1580,
 97,
 4288,
 3606,
 2736,
 3607,
 1585,
 3608,
 4790,
 5151,
 1589,
 1590,
 108,
 4792,
 110,
 2742,
 6226,
 113,
 114,
 115,
 116,
 6227,
 1595,
 119,
 120,
 121,
 4795,
 5630,
 124,
 4797,
 126,
 4798,
 128,
 1600,
 1601,
 131,
 1602,
 2749,
 4799,
 6228,
 1606,
 137,
 138,
 3620,
 140,
 4801,
 5932,
 3623,
 1611,
 6032,
 146,
 4804,
 148,
 2758,
 6229,
 3

In [16]:
df_full.loc[df_full['problem'].isin(idx),:]

,problem,feedback,n,block,ha,pha,la,hb,phb,lb,lotshapeb,lotnumb,amb,corr,brate,brate_std,B,A
6,7,False,15,1,-6,1.00,-6,24,0.20,-5,1,2,False,0,0.866667,0.351866,"[[0.8, -5.0], [0.1, 23.5], [0.1, 24.5]]","[[1.0, -6.0], [0.0, -6.0]]"
7,7,True,16,3,-6,1.00,-6,24,0.20,-5,1,2,False,0,0.962500,0.150000,"[[0.8, -5.0], [0.1, 23.5], [0.1, 24.5]]","[[1.0, -6.0], [0.0, -6.0]]"
8,8,False,15,1,27,1.00,27,89,0.50,-24,0,1,False,0,0.453333,0.403320,"[[0.5, -24.0], [0.5, 89.0]]","[[1.0, 27.0], [0.0, 27.0]]"
9,8,True,16,2,27,1.00,27,89,0.50,-24,0,1,False,0,0.612500,0.434933,"[[0.5, -24.0], [0.5, 89.0]]","[[1.0, 27.0], [0.0, 27.0]]"
11,10,False,15,1,1,1.00,1,59,0.05,0,0,1,True,0,0.773333,0.406143,"[[0.9500000000000001, 0.0], [0.05, 59.0]]","[[1.0, 1.0], [0.0, 1.0]]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8293,6734,True,18,5,77,0.01,30,116,0.40,-12,2,5,True,0,0.688889,0.292834,"[[0.6000000000000001, -12.0], [0.2, 112.0], [0...","[[0.01, 77.0], [0.99, 30.0]]"
8296,6737,False,16,1,26,1.00,26,41,0.60,7,0,1,False,0,0.450000,0.287518,"[[0.4, 7.0], [0.6000000000000001, 41.0]]","[[1.0, 26.0], [0.0, 26.0]]"
8297,6737,True,16,3,26,1.00,26,41,0.60,7,0,1,False,0,0.600000,0.309839,"[[0.4, 7.0], [0.6000000000000001, 41.0]]","[[1.0, 26.0], [0.0, 26.0]]"
8299,6739,False,16,1,-1,0.95,-4,10,0.75,-17,2,3,False,0,0.787500,0.247319,"[[0.25, -17.0], [0.375, 8.0], [0.1875, 10.0], ...","[[0.9500000000000001, -1.0], [0.05, -4.0]]"


In [17]:
df_full['feedback'].value_counts()

feedback
True     12188
False     2380
Name: count, dtype: int64

In [18]:
count = (
    df_full
    .groupby("problem")["feedback"]
    .nunique()
    .eq(2)
    .sum()
)

print(count)


1562


## Feature Engineering

In [19]:
numeric_cols = df_full.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if col != 'problem':
        df_full[col] = df_full[col].round(2)
def round_gamble_column(gamble_list, decimals=2):
    if isinstance(gamble_list, str):
        import ast
        gamble_list = ast.literal_eval(gamble_list)

    return [(round(p, decimals), round(r, decimals)) for p, r in gamble_list]

def calculate_gamble_stats(gamble_list):
    if isinstance(gamble_list, str):
        import ast
        gamble_list = ast.literal_eval(gamble_list)

    probs = [x[0] for x in gamble_list]
    rewards = [x[1] for x in gamble_list]

    ev = sum(p * r for p, r in zip(probs, rewards))

    variance = sum(p * (r - ev)**2 for p, r in zip(probs, rewards))
    std_dev = np.sqrt(variance)

    return round(ev, 2), round(std_dev, 2)
df_full['A'] = df_full['A'].apply(round_gamble_column)
df_full['B'] = df_full['B'].apply(round_gamble_column)

df_full['A_ev'], df_full['A_std'] = zip(*df_full['A'].apply(calculate_gamble_stats))

df_full['B_ev'], df_full['B_std'] = zip(*df_full['B'].apply(calculate_gamble_stats))

# Difference in variability/risk features
df_full['std_diff'] = df_full['A_std'] - df_full['B_std']
df_full['variance_diff'] = df_full['A_std']**2 - df_full['B_std']**2

# Function to check for extreme outcomes in a gamble
def has_extreme_outcomes(gamble_list, loss_threshold=-20, gain_threshold=50):
    """
    Check if a gamble has extreme outcomes (large losses or gains).

    Parameters:
    -----------
    gamble_list : list
        List of (probability, reward) tuples
    loss_threshold : float
        Threshold for large loss (default: -20)
    gain_threshold : float
        Threshold for large gain (default: 50)

    Returns:
    --------
    tuple : (has_large_loss, has_large_gain)
    """
    if isinstance(gamble_list, str):
        import ast
        gamble_list = ast.literal_eval(gamble_list)

    rewards = [x[1] for x in gamble_list]
    has_large_loss = any(r <= loss_threshold for r in rewards)
    has_large_gain = any(r >= gain_threshold for r in rewards)

    return has_large_loss, has_large_gain

# Calculate extreme outcome indicators for options A and B
df_full['A_has_large_loss'], df_full['A_has_large_gain'] = zip(*df_full['A'].apply(
    lambda x: has_extreme_outcomes(x, loss_threshold=-20, gain_threshold=50)
))
df_full['B_has_large_loss'], df_full['B_has_large_gain'] = zip(*df_full['B'].apply(
    lambda x: has_extreme_outcomes(x, loss_threshold=-20, gain_threshold=50)
))

# Additional Naïve features
# Difference in expected value
df_full['ev_diff'] = df_full['A_ev'] - df_full['B_ev']

# Difference in worst outcomes (minima)
def get_min_outcome(gamble_list):
    """Get the minimum (worst) outcome from a gamble."""
    if isinstance(gamble_list, str):
        import ast
        gamble_list = ast.literal_eval(gamble_list)
    rewards = [x[1] for x in gamble_list]
    return min(rewards)

df_full['A_min'] = df_full['A'].apply(get_min_outcome)
df_full['B_min'] = df_full['B'].apply(get_min_outcome)
df_full['min_diff'] = df_full['A_min'] - df_full['B_min']

# Probability of positive payoff
def prob_positive(gamble_list):
    """Calculate probability of getting a positive payoff."""
    if isinstance(gamble_list, str):
        import ast
        gamble_list = ast.literal_eval(gamble_list)
    return sum(p for p, r in gamble_list if r > 0)

df_full['A_prob_positive'] = df_full['A'].apply(prob_positive)
df_full['B_prob_positive'] = df_full['B'].apply(prob_positive)
df_full['prob_positive_diff'] = df_full['A_prob_positive'] - df_full['B_prob_positive']

# Psychological insight (BEAST tool) features
# Probability option A beats option B in a single draw: P(XA > XB)
def prob_A_beats_B(gamble_A, gamble_B):
    """
    Calculate probability that option A beats option B in a single draw.
    Computed by convolving the outcome distributions.
    """
    if isinstance(gamble_A, str):
        import ast
        gamble_A = ast.literal_eval(gamble_A)
    if isinstance(gamble_B, str):
        import ast
        gamble_B = ast.literal_eval(gamble_B)

    prob_A_beats = 0.0
    for p_A, r_A in gamble_A:
        for p_B, r_B in gamble_B:
            if r_A > r_B:
                prob_A_beats += p_A * p_B

    return prob_A_beats

df_full['prob_A_beats_B'] = df_full.apply(
    lambda row: prob_A_beats_B(row['A'], row['B']), axis=1
)

# Regret risk: Expected regret if choosing A: E[max(0, XB - XA)]
def expected_regret(gamble_A, gamble_B):
    """
    Calculate expected regret if choosing option A.
    Regret = max(0, XB - XA)
    """
    if isinstance(gamble_A, str):
        import ast
        gamble_A = ast.literal_eval(gamble_A)
    if isinstance(gamble_B, str):
        import ast
        gamble_B = ast.literal_eval(gamble_B)

    expected_regret_val = 0.0
    for p_A, r_A in gamble_A:
        for p_B, r_B in gamble_B:
            regret = max(0, r_B - r_A)
            expected_regret_val += p_A * p_B * regret

    return expected_regret_val

df_full['expected_regret'] = df_full.apply(
    lambda row: expected_regret(row['A'], row['B']), axis=1
)

# Probability of regret: P(XB > XA)
def prob_regret(gamble_A, gamble_B):
    """Calculate probability of regret: P(XB > XA)."""
    if isinstance(gamble_A, str):
        import ast
        gamble_A = ast.literal_eval(gamble_A)
    if isinstance(gamble_B, str):
        import ast
        gamble_B = ast.literal_eval(gamble_B)

    prob_regret_val = 0.0
    for p_A, r_A in gamble_A:
        for p_B, r_B in gamble_B:
            if r_B > r_A:
                prob_regret_val += p_A * p_B

    return prob_regret_val

df_full['prob_regret'] = df_full.apply(
    lambda row: prob_regret(row['A'], row['B']), axis=1
)

# Worst-case sensitivity: Probability of hitting the worst outcome
def prob_worst_outcome(gamble_list):
    """Calculate probability of hitting the worst (minimum) outcome."""
    if isinstance(gamble_list, str):
        import ast
        gamble_list = ast.literal_eval(gamble_list)

    rewards = [x[1] for x in gamble_list]
    min_reward = min(rewards)

    return sum(p for p, r in gamble_list if r == min_reward)

df_full['A_prob_worst'] = df_full['A'].apply(prob_worst_outcome)
df_full['B_prob_worst'] = df_full['B'].apply(prob_worst_outcome)
df_full['prob_worst_diff'] = df_full['A_prob_worst'] - df_full['B_prob_worst']

# Best-sign sensitivity: P(XA > 0) - P(XB > 0) (already calculated as prob_positive_diff above)

df_full.head(10)

,problem,feedback,n,block,ha,pha,la,hb,phb,lb,...,min_diff,A_prob_positive,B_prob_positive,prob_positive_diff,prob_A_beats_B,expected_regret,prob_regret,A_prob_worst,B_prob_worst,prob_worst_diff
0,1,True,15,2,26,0.95,-1,23,0.05,21,...,-22.0,0.95,1.00,-0.05,0.95,1.105,0.05,0.05,0.95,-0.90
1,2,True,15,4,14,0.60,-18,8,0.25,-5,...,-13.0,0.60,0.25,0.35,0.60,6.500,0.40,0.40,0.75,-0.35
2,3,True,17,4,2,0.50,0,1,1.00,1,...,-1.0,0.50,1.00,-0.50,0.50,0.500,0.50,0.50,1.00,-0.50
3,4,True,18,3,37,0.05,8,87,0.25,-31,...,39.0,1.00,0.24,0.76,0.75,18.612,0.24,0.95,0.75,0.20
4,5,False,15,1,26,1.00,26,45,0.75,-36,...,62.0,1.00,0.76,0.24,0.25,14.520,0.76,1.00,0.25,0.75
5,6,True,15,4,28,1.00,28,33,0.01,28,...,0.0,1.00,1.00,0.00,0.00,0.030,0.01,1.00,0.99,0.01
6,7,False,15,1,-6,1.00,-6,24,0.20,-5,...,-1.0,0.00,0.20,-0.20,0.00,6.800,1.00,1.00,0.80,0.20
7,7,True,16,3,-6,1.00,-6,24,0.20,-5,...,-1.0,0.00,0.20,-0.20,0.00,6.800,1.00,1.00,0.80,0.20
8,8,False,15,1,27,1.00,27,89,0.50,-24,...,51.0,1.00,0.50,0.50,0.50,31.000,0.50,1.00,0.50,0.50
9,8,True,16,2,27,1.00,27,89,0.50,-24,...,51.0,1.00,0.50,0.50,0.50,31.000,0.50,1.00,0.50,0.50


# Modelling

In [20]:
class EvaluationResult:
    """
    Class to store experiment evaluation results.

    Attributes:
    -----------
    best_estimator : sklearn estimator
        The best fitted estimator (pipeline)
    best_params : dict
        Best hyperparameters found (None if no hyperparameter tuning was performed)
    predictions : array-like
        Predictions on test set
    metrics : dict
        Dictionary containing evaluation metrics (MAE, RMSE, MSE, R2)
    """
    def __init__(self, best_estimator, best_params, predictions, metrics):
        self.best_estimator = best_estimator
        self.best_params = best_params
        self.predictions = predictions
        self.metrics = metrics

    def __repr__(self):
        return (f"EvaluationResult(\n"
                f"  best_params={self.best_params},\n"
                f"  metrics={self.metrics}\n"
                f")")


In [21]:

TARGET_COL = 'brate'
# Use only numeric/bool columns so StandardScaler and models get no list-valued columns (e.g. A, B)
X = df_full.drop(columns=[TARGET_COL]).select_dtypes(include=[np.number, "bool"])

In [46]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV


def run_experiment(model, X, y, params=None, use_randomized=True, cv_folds=3,
                   n_iter=20, verbose=1, sample_size=None, run_full_pipeline=True):
    """
    Run experiment with optional hyperparameter tuning.

    Parameters:
    -----------
    model : sklearn estimator
        The model to use
    X : DataFrame
        Features
    y : Series
        Target variable
    params : dict, optional
        Parameter grid for hyperparameter tuning
    use_randomized : bool, default=True
        If True, use RandomizedSearchCV (faster). If False, use GridSearchCV (exhaustive)
    cv_folds : int, default=3
        Number of cross-validation folds (lower = faster)
    n_iter : int, default=20
        Number of iterations for RandomizedSearchCV (ignored if use_randomized=False)
    verbose : int, default=1
        Verbosity level (0=silent, 1=progress, 2=detailed)
    sample_size : float or int, optional
        If float (0-1), sample that fraction of training data for faster grid search.
        If int, sample that many rows. If None, use all data.

    Returns:
    --------
    EvaluationResult
        Object containing best_estimator, best_params, predictions, and metrics
    """
    if run_full_pipeline:
        num_cols = X.select_dtypes(include=[np.number, "bool"]).columns.tolist()

        pipeline = Pipeline(steps=[
            ("prep", ColumnTransformer(
                transformers=[
                    ("num", Pipeline(steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]), num_cols),
                ],
                remainder="drop"
            )),
            ("reg", model)
        ])
    else:
        pipeline = model

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    best_params = None
    best_estimator = None

    best_params = None
    best_estimator = None

    # Optionally sample training data for faster grid search
    if sample_size is not None and params is not None:
        if isinstance(sample_size, float):
            n_samples = int(len(X_train) * sample_size)
        else:
            n_samples = min(sample_size, len(X_train))
        sample_idx = np.random.RandomState(42).choice(len(X_train), n_samples, replace=False)
        X_train_sample = X_train.iloc[sample_idx] if hasattr(X_train, 'iloc') else X_train[sample_idx]
        y_train_sample = y_train.iloc[sample_idx] if hasattr(y_train, 'iloc') else y_train[sample_idx]
        print(f"Using {n_samples}/{len(X_train)} samples for grid search...")
    else:
        X_train_sample = X_train
        y_train_sample = y_train

    # If params provided, use hyperparameter search
    if params is not None:
        # Prefix params with "reg__" since the model is in the "reg" step of the pipeline
        param_grid = {f"reg__{k}": v for k, v in params.items()}

        if use_randomized:
            search = RandomizedSearchCV(
                estimator=pipeline,
                param_distributions=param_grid,
                n_iter=n_iter,
                cv=cv_folds,
                scoring='neg_mean_absolute_error',
                n_jobs=-1,
                random_state=42,
                verbose=verbose
            )
            print(f"Running RandomizedSearchCV with {n_iter} iterations, {cv_folds}-fold CV...")
        else:
            search = GridSearchCV(
                estimator=pipeline,
                param_grid=param_grid,
                cv=cv_folds,
                scoring='neg_mean_absolute_error',
                n_jobs=-1,
                verbose=verbose
            )
            total_combinations = 1
            for v in param_grid.values():
                total_combinations *= len(v)
            print(f"Running GridSearchCV with {total_combinations} combinations, {cv_folds}-fold CV...")

        search.fit(X_train_sample, y_train_sample)

        # Refit best model on full training data if we sampled
        if sample_size is not None:
            print("Refitting best model on full training data...")
            best_params = search.best_params_
            pipeline.set_params(**best_params)
            pipeline.fit(X_train, y_train)
            pred = pipeline.predict(X_test)
            best_estimator = pipeline
        else:
            pred = search.predict(X_test)
            best_estimator = search.best_estimator_

        best_params = search.best_params_
        print("Best parameters:", best_params)
        print("Best CV score (MAE):", -search.best_score_)
    else:
        # No grid search, just fit the pipeline
        pipeline.fit(X_train, y_train)
        pred = pipeline.predict(X_test)
        best_estimator = pipeline
        best_estimator = pipeline

    # Calculate metrics
    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    mse = mean_squared_error(y_test, pred)
    r2 = r2_score(y_test, pred)

    metrics = {
        'MAE': mae,
        'RMSE': rmse,
        'MSE': mse,
        'R2': r2
    }

    print("MAE:", mae)
    print("RMSE:", rmse)
    print("MSE:", mse)
    print("R^2:", r2)

    # Return EvaluationResult object
    return EvaluationResult(
        best_estimator=best_estimator,
        best_params=best_params,
        predictions=pred,
        metrics=metrics
    )

In [23]:
y = df_full[TARGET_COL]

In [24]:
model1 = Ridge(random_state=42)
params = {'alpha': [0.01, 0.1, 1, 10], 'solver': ['auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga']}
# Use run_experiment function
result1 = run_experiment(model1, X, y, params=params)

# Extract results from EvaluationResult object
pipeline = result1.best_estimator
pred1 = result1.predictions
best_params1 = result1.best_params
metrics1 = result1.metrics


Running RandomizedSearchCV with 20 iterations, 3-fold CV...
Fitting 3 folds for each of 20 candidates, totalling 60 fits


Best parameters: {'reg__solver': 'lsqr', 'reg__alpha': 0.01}
Best CV score (MAE): 0.10128052545572684
MAE: 0.10166871017539937
RMSE: 0.12680948057680436
MSE: 0.01608064436415892
R^2: 0.6717091479986598


In [25]:
# Print metrics from EvaluationResult
print("MAE:", metrics1['MAE'])
print("RMSE:", metrics1['RMSE'])
print("MSE:", metrics1['MSE'])
print("R^2:", metrics1['R2'])


MAE: 0.10166871017539937
RMSE: 0.12680948057680436
MSE: 0.01608064436415892
R^2: 0.6717091479986598


In [26]:
from sklearn.ensemble import RandomForestRegressor
model2 = RandomForestRegressor(random_state=42)
params = {  'n_estimators': [100, 200, 300],
            'max_depth': [None, 10, 20],
            'min_samples_split': [2, 5, 10]
        }

# Use run_experiment function
result2 = run_experiment(model2, X, y, params=params)

# Extract results from EvaluationResult object
pipeline = result2.best_estimator
pred2 = result2.predictions
best_params2 = result2.best_params
metrics2 = result2.metrics


Running RandomizedSearchCV with 20 iterations, 3-fold CV...
Fitting 3 folds for each of 20 candidates, totalling 60 fits


KeyboardInterrupt: 

In [ ]:
# Print metrics from EvaluationResult
print("MAE:", metrics2['MAE'])
print("RMSE:", metrics2['RMSE'])
print("MSE:", metrics2['MSE'])
print("R^2:", metrics2['R2'])

MAE: 0.06861052390757262
RMSE: 0.09335647875822208
MSE: 0.00871543212613437
R^2: 0.8220720156820559


In [27]:
from sklearn.svm import SVR
model3 = SVR()
params = {'kernel': ['linear', 'rbf', 'sigmoid'], 'C': [0.1, 1, 10]}

# Use run_experiment function
result3 = run_experiment(model3, X, y, params=params)

# Extract results from EvaluationResult object
pipeline = result3.best_estimator
pred3 = result3.predictions
best_params3 = result3.best_params
metrics3 = result3.metrics


Running RandomizedSearchCV with 20 iterations, 3-fold CV...
Fitting 3 folds for each of 9 candidates, totalling 27 fits


d:\Users\anast\AppData\Local\Lib\site-packages\sklearn\model_selection\_search.py:317: UserWarning: The total space of parameters 9 is smaller than n_iter=20. Running 9 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


KeyboardInterrupt: 

In [ ]:
# Print metrics from EvaluationResult
print("MAE:", metrics3['MAE'])
print("RMSE:", metrics3['RMSE'])
print("MSE:", metrics3['MSE'])
print("R^2:", metrics3['R2'])


MAE: 0.07163195658784478
RMSE: 0.09180235466968238
MSE: 0.008427672322898155
R^2: 0.827946712543502


In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
model4 =  GradientBoostingRegressor(random_state=42)

params = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.1, 1],
    'max_depth': [3, 4, 5]
 }

result4 = run_experiment(model4, X, y, params=params)


Running RandomizedSearchCV with 20 iterations, 3-fold CV...
Fitting 3 folds for each of 20 candidates, totalling 60 fits
Best parameters: {'reg__n_estimators': 200, 'reg__max_depth': 5, 'reg__learning_rate': 0.1}
Best CV score (MAE): 0.06705076092451963
MAE: 0.06435997127611215
RMSE: 0.08596115246559434
MSE: 0.007389319733213155
R^2: 0.8491449710601355


In [ ]:
from sklearn.ensemble import AdaBoostRegressor
model5 = AdaBoostRegressor(random_state=42)
params = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.1, 1],
    'loss': ['linear', 'square', 'exponential']
}

result5 = run_experiment(model5, X, y, params=params)

Running RandomizedSearchCV with 20 iterations, 3-fold CV...
Fitting 3 folds for each of 20 candidates, totalling 60 fits
Best parameters: {'reg__n_estimators': 300, 'reg__loss': 'square', 'reg__learning_rate': 1}
Best CV score (MAE): 0.09896369882273963
MAE: 0.09856645069865477
RMSE: 0.11774698920902354
MSE: 0.013864353467789909
R^2: 0.7169553464826875


In [ ]:
from sklearn.neural_network import MLPRegressor
model6 = MLPRegressor(random_state=42)
params = {
    'hidden_layer_sizes': [(100,), (200,), (300,)],
    'activation': ['relu', 'tanh', 'logistic'],
    'alpha': [0.0001, 0.001, 0.01]
}
result6 = run_experiment(model6, X, y, params=params)

Running RandomizedSearchCV with 20 iterations, 3-fold CV...
Fitting 3 folds for each of 20 candidates, totalling 60 fits
Best parameters: {'reg__hidden_layer_sizes': (200,), 'reg__alpha': 0.01, 'reg__activation': 'tanh'}
Best CV score (MAE): 0.07623227736871845
MAE: 0.07185446934181687
RMSE: 0.09391083388874509
MSE: 0.008819244721679474
R^2: 0.8199526525105195


In [ ]:
from sklearn.ensemble import StackingRegressor
model7 = StackingRegressor(
    estimators=[
        ('rf', RandomForestRegressor(random_state=42)),
        ('svr', SVR()),
        ('gbr', GradientBoostingRegressor(random_state=42))
    ],
    final_estimator=Ridge(random_state=42),
    cv=5
)
result7 = run_experiment(model7, X, y, params=None)

MAE: 0.06594751146770714
RMSE: 0.08659865514252878
MSE: 0.007499327072494627
R^2: 0.8468991404627244


In [ ]:
from xgboost import XGBRegressor
xgb = XGBRegressor(random_state=42)
params = {
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.01, 0.1, 1],
    "max_depth": [3, 4, 5]
}
result8 = run_experiment(xgb, X, y, params=params, use_randomized=False, cv_folds=3)

In [ ]:
from lightgbm import LGBMRegressor
lgb = LGBMRegressor( random_state=42)


params = {
    "learning_rate": [0.05, 0.1],
    "num_leaves": [31, 63],
    "max_depth": [6, -1],
    "min_child_samples": [20, 60],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
}

result9 = run_experiment(lgb, X, y, params=params, use_randomized=False, cv_folds=3)

Running GridSearchCV with 64 combinations, 3-fold CV...
Fitting 3 folds for each of 64 candidates, totalling 192 fits
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003515 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4176
[LightGBM] [Info] Number of data points in the train set: 11654, number of used features: 38
[LightGBM] [Info] Start training from score 0.518630
Best parameters: {'reg__colsample_bytree': 0.8, 'reg__learning_rate': 0.1, 'reg__max_depth': -1, 'reg__min_child_samples': 20, 'reg__num_leaves': 63, 'reg__subsample': 0.8}
Best CV score (MAE): 0.06584743116901916
MAE: 0.06325674491804104
RMSE: 0.08497270847681476
MSE: 0.007220361185885747
R^2: 0.8525943070568106


d:\Users\anast\AppData\Local\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [47]:
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge

# Use only the regressor step: run_experiment's pipeline does prep once; base estimators receive arrays.
estimators = [
    ('xgb', result8.best_estimator.named_steps['reg']),
    ('lgb', result9.best_estimator.named_steps['reg']),
]


stack = StackingRegressor(estimators=estimators, final_estimator=Ridge(1.0), cv=5)
result10 = run_experiment(stack, X, y, params=None)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001851 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4176
[LightGBM] [Info] Number of data points in the train set: 11654, number of used features: 38
[LightGBM] [Info] Start training from score 0.518630
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000947 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4134
[LightGBM] [Info] Number of data points in the train set: 9323, number of used features: 38
[LightGBM] [Info] Start training from score 0.517852


d:\Users\anast\AppData\Local\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000962 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4138
[LightGBM] [Info] Number of data points in the train set: 9323, number of used features: 38
[LightGBM] [Info] Start training from score 0.517711


d:\Users\anast\AppData\Local\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001601 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4126
[LightGBM] [Info] Number of data points in the train set: 9323, number of used features: 38
[LightGBM] [Info] Start training from score 0.520194


...

d:\Users\anast\AppData\Local\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000936 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4135
[LightGBM] [Info] Number of data points in the train set: 9324, number of used features: 38
[LightGBM] [Info] Start training from score 0.517865


d:\Users\anast\AppData\Local\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


MAE: 0.0624310039497292
RMSE: 0.08381112645684494
MSE: 0.007024304917965254
R^2: 0.856596850597869


d:\Users\anast\AppData\Local\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


### Neural Network

In [51]:
X = X.select_dtypes(include=[np.number, "bool"])

In [52]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [53]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [56]:
X_tr_torch = torch.FloatTensor(X_train_scaled)
y_tr_torch = torch.FloatTensor(y_train.values).unsqueeze(1)
X_te_torch = torch.FloatTensor(X_test_scaled)
y_te_torch = torch.FloatTensor(y_test.values).unsqueeze(1)

In [57]:
train_dataset = TensorDataset(X_tr_torch, y_tr_torch)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)

In [ ]:
class Choices13kNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
input_dim = X_train.shape[1]
model = Choices13kNet(input_dim)
criterion = nn.MSELoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=20, factor=0.5)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

best_val_loss = float('inf')
patience_counter = 0
history = {'train_loss': [], 'val_loss': []}

for epoch in range(500):
    model.train()
    train_loss = 0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        pred = model(batch_x)
        loss = criterion(pred, batch_y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()

    model.eval()
    with torch.no_grad():
        val_pred = model(X_te_torch.to(device))
        val_loss = criterion(val_pred.cpu(), y_te_torch).item()

    scheduler.step(val_loss)
    history['train_loss'].append(train_loss / len(train_loader))
    history['val_loss'].append(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), 'best_choices13k.pt')
    else:
        patience_counter += 1
        if patience_counter > 50:
            break

    if epoch % 25 == 0:
        print(f"Epoch {epoch}: Val MSE {val_loss:.5f}")


In [ ]:
plt.figure(figsize=(8,5))
plt.plot(history['train_loss'], label='Training Loss')
plt.plot(history['val_loss'], label='Validation Loss')
plt.title('Training vs Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

AttributeError: 'dict' object has no attribute 'history'

<Figure size 800x500 with 0 Axes>

In [ ]:
model.load_state_dict(torch.load('best_choices13k.pt'))
model.eval()
with torch.no_grad():
    final_pred = model(X_te_torch.to(device)).cpu().numpy().flatten()

mse = mean_squared_error(y_test, final_pred)
mae = mean_absolute_error(y_test, final_pred)
r2 = r2_score(y_test, final_pred)

print(f"MSE: {mse:.5f}, MAE: {mae:.4f}, R2: {r2:.3f}")
print(f"PyTorch NN - MSE: {mse:.5f}, MAE: {mae:.4f}")

In [ ]:
class FTTransformer(nn.Module):
    """
    FT-Transformer: feature tokenizer + transformer for tabular data.
    - Numerical features: per-feature linear embedding (scalar -> d_model).
    - Categorical features: per-feature embedding table (category index -> d_model).
    - [CLS] token + feature tokens -> Transformer encoder -> predict from [CLS].
    """
    def __init__(
        self,
        n_num_features: int,
        cat_cardinalities: list,
        d_model: int = 64,
        n_heads: int = 4,
        n_layers: int = 2,
        d_ff: int = 128,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.n_num = n_num_features
        self.cat_cardinalities = cat_cardinalities or []
        self.d_model = d_model
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"

        # Per-feature numerical embedding: each feature -> d_model
        self.num_embeds = nn.ModuleList([
            nn.Linear(1, d_model) for _ in range(n_num_features)
        ])

        # Per-feature categorical embedding
        self.cat_embeds = nn.ModuleList([
            nn.Embedding(card, d_model) for card in self.cat_cardinalities
        ])

        n_tokens = 1 + n_num_features + len(self.cat_cardinalities)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_ff,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=False,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, 1),
            nn.Sigmoid(),
        )

    def forward(self, x_num: torch.Tensor, x_cat: torch.Tensor = None) -> torch.Tensor:
        """
        x_num: (B, n_num) float
        x_cat: (B, n_cat) long, optional; if None, only numerical tokens are used.
        """
        B = x_num.size(0)
        tokens = []
        # [CLS]
        cls = self.cls_token.expand(B, -1, -1)
        tokens.append(cls)
        # Numerical tokens: per-feature linear
        for i in range(self.n_num):
            fi = x_num[:, i : i + 1]
            tokens.append(self.num_embeds[i](fi).unsqueeze(1)) # Corrected line
        if x_cat is not None and len(self.cat_embeds) > 0:
            for i in range(x_cat.size(1)):
                # Re-corrected to use cat_embeds and x_cat
                tokens.append(self.cat_embeds[i](x_cat[:, i]).unsqueeze(1))
        x = torch.cat(tokens, dim=1)
        x = self.transformer(x)
        cls_out = x[:, 0]
        return self.head(cls_out)

In [ ]:
# FT-Transformer: use same X_train_scaled / X_test_scaled (all numerical + bool as numeric)
n_num_features = X_train.shape[1]
cat_cardinalities = []  # optional: e.g. [2, 3] if you treat Feedback, LotShapeB as categorical

ft_model = FTTransformer(
    n_num_features=n_num_features,
    cat_cardinalities=cat_cardinalities,
    d_model=64,
    n_heads=4,
    n_layers=2,
    d_ff=128,
    dropout=0.1,
)
ft_optimizer = optim.AdamW(ft_model.parameters(), lr=1e-3, weight_decay=0.01)
ft_criterion = nn.MSELoss()
ft_scheduler = optim.lr_scheduler.ReduceLROnPlateau(ft_optimizer, patience=20, factor=0.5)

ft_train_dataset = TensorDataset(X_tr_torch, y_tr_torch)
ft_train_loader = DataLoader(ft_train_dataset, batch_size=128, shuffle=True)

In [ ]:
device_ft = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ft_model.to(device_ft)
best_ft_val = float("inf")
patience_ft = 0
N_EPOCHS_FT = 200
history = {'train_loss': [], 'val_loss': []}
for epoch in range(N_EPOCHS_FT):
    ft_model.train()
    train_loss_ft = 0.0
    for batch_x, batch_y in ft_train_loader:
        batch_x = batch_x.to(device_ft)
        batch_y = batch_y.to(device_ft)
        ft_optimizer.zero_grad()
        pred = ft_model(batch_x, None)
        loss = ft_criterion(pred, batch_y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(ft_model.parameters(), 1.0)
        ft_optimizer.step()
        train_loss_ft += loss.item()
    ft_model.eval()
    with torch.no_grad():
        val_pred = ft_model(X_te_torch.to(device_ft), None)
        val_loss = ft_criterion(val_pred.cpu(), y_te_torch).item()
    ft_scheduler.step(val_loss)
    history['train_loss'].append(train_loss_ft / len(ft_train_loader))
    history['val_loss'].append(val_loss)
    if val_loss < best_ft_val:
        best_ft_val = val_loss
        patience_ft = 0
        torch.save(ft_model.state_dict(), "best_ft_transformer.pt")
    else:
        patience_ft += 1
        if patience_ft > 50:
            break
    if epoch % 25 == 0:
        print(f"FT-Transformer Epoch {epoch}: Val MSE {val_loss:.5f}")

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(history['train_loss'], label='Training Loss')
plt.plot(history['val_loss'], label='Validation Loss')
plt.title('Training vs Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
ft_model.load_state_dict(torch.load("best_ft_transformer.pt", map_location=device_ft))
ft_model.eval()
with torch.no_grad():
    ft_pred = ft_model(X_te_torch.to(device_ft), None).cpu().numpy().flatten()

ft_mae = mean_absolute_error(y_test, ft_pred)
ft_rmse = np.sqrt(mean_squared_error(y_test, ft_pred))
ft_r2 = r2_score(y_test, ft_pred)
print(f"FT-Transformer — MAE: {ft_mae:.4f}, RMSE: {ft_rmse:.4f}, R²: {ft_r2:.3f}")